# [3.2] SigLIP Batch 2 (Chuẩn YAML + Tải Trực Tiếp Từ Modal) → Volume `keyframes-btc`

`docs/triet/ke-hoach-offline-ingest.md` §2.1, §2.3, §7 · config `config/ingest_shards_batch2_btc_images.yaml`

Chạy **sau** `02-keyframes-btc-import.ipynb`. Cùng bundle, cùng volume, **cùng cách chia người** —
đổi `MEMBER` ở ô đầu là ra đúng phần việc bạn đã nhập ở lượt trước.

### Ba tầng, và tầng nào KHÔNG có ở đây

| Tầng | Model | Ra artifact |
|---|---|---|
| `n4_siglip` | `google/siglip-so400m-patch14-384` (tháp ảnh) | `siglip_vectors` |
| `n5_ocr_easyocr` | easyocr vi+en | `ocr_text` |
| `n5_ocr_qwen` | `Qwen/Qwen2-VL-7B-Instruct` 4-bit | `sviptr_ocr` |

`n4_objects` **không có**: BTC phát hành sẵn nhãn object và notebook 02 đã nạp chúng. Chạy
detector của ta lên chính bộ keyframe này là phép đo khác (§3.6) và là việc khác.

`n6_recap` / `n6_bge_m3` **không có**: Lead đã hoãn tầng ReCap.

### Vì sao ảnh tải lại từ zip chứ không kéo từ Volume

Volume ở đường BTC là **ghi một chiều**. Kéo ảnh về là ~177.000 lần `read_file`; tải lại gói
`Keyframes_*.zip` là 1,4–5,7 GB một lần, có resume. Cùng dữ liệu, khác hai bậc chi phí.

### Ba model cùng nằm trong VRAM một lúc — có chủ ý

SigLIP ~1,1 GB + easyocr ~0,1 GB + Qwen2-VL 4-bit ~5,5 GB ≈ **6,7 GB / 15 GB của T4**. Giữ cả ba
để mỗi model chỉ nạp **một lần cho cả phiên**; nạp lại theo từng video tốn **5–16 giờ mỗi người
chỉ để nạp** (đo trên số video thật của từng người).

> Nếu gặp OOM: đặt `KEEP_MODELS = False` ở ô đầu. Notebook sẽ thả model sau mỗi video — đúng hành
> vi cũ của §4.2, đổi lại là thời gian nạp.

### `SKIP_STAGES` — bỏ qua một tầng cho LƯỢT CHẠY NÀY

Mặc định đang là `["n5_ocr_qwen"]`: chạy SigLIP + easyocr trước, để Qwen lại sau. Muốn chỉ chạy
SigLIP thì thêm `"n5_ocr_easyocr"` vào danh sách.

Bỏ qua ở đây **không cắm marker**, nên lần chạy sau vẫn thấy tầng đó là việc còn lại và chạy tiếp
từ chỗ đứt. Đó là khác biệt so với việc **xoá tầng khỏi yaml**:

| | `SKIP_STAGES` | xoá khỏi `config/…_btc_images.yaml` |
|---|---|---|
| lượt này | bỏ qua | bỏ qua |
| lần chạy sau | vẫn là việc còn lại | biến mất khỏi kế hoạch |
| `merge_bundle.py` | **vẫn đòi đủ mảnh** | không đòi nữa |

Nên chừng nào `n5_ocr_qwen` còn trong yaml thì **`merge_bundle.py` sẽ từ chối bundle** cho tới khi
nó chạy xong cả 873 video. Đó đúng là điều ta muốn lúc này (hoãn, không bỏ). Nếu Lead quyết **bỏ
hẳn** Qwen — có cơ sở: đo được 10,8 giây/ảnh, tức ~532 giờ GPU cho batch 1, và lượt 4 đã sinh
`paddle_ocr` + `viet_ocr` thay chỗ — thì phải xoá tầng khỏi yaml, không phải để nó ở `SKIP_STAGES`.

> ⚠️ Ô Step 0b từng có dòng `MY_VIDEOS = MY_VIDEOS[:1]` — di sản của một lần chạy thử, nó cắt phần
> việc của mọi người xuống **một video**. Đã xoá.


## Dùng lại private Kaggle cache

Attach cùng Dataset đã dùng cho notebook 02. Kaggle thường tự giải nén archive thành thư mục;
notebook nhận cả layout đã giải nén lẫn ZIP nguyên bản trong `/kaggle/input`, giữ cache read-only
và chỉ tải từ website khi cả hai dạng đều thiếu. Không copy 15 GiB sang `/kaggle/working`.


## Step 0a — repo + gói

In [ ]:
MEMBER      = "missing1"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch2_missing.yaml"  # NOTE: File này phải tồn tại hoặc bạn phải sửa lại tên cho đúng
TRANSPORT   = "modal"
VOLUME      = "keyframes-batch-2-v2"  # CÙNG volume với notebook 02
VOL_VERSION = 2
KEEP_MODELS = True             # False = thả model sau mỗi video (xem ô mô tả trên)

# ── CHẠY THỬ / CHẠY LẠI ───────────────────────────────────────────────────
# ONLY_VIDEOS: [] = cả phần việc của bạn. Điền video_id để chỉ chạy bấy nhiêu.
# REDO_STAGES: chạy lại DÙ marker đã báo xong. Cần khi đổi prompt hoặc tham
#              số — nếu không thì `StageStatus` đọc marker từ Volume và bỏ
#              qua đúng những việc ta muốn làm lại. Nó GHI ĐÈ part file và
#              marker cũ, nên chỉ khai khi thật sự muốn thay dữ liệu.
ONLY_VIDEOS = []
REDO_STAGES = []

# SKIP_STAGES: bo qua tang NAY LUOT NAY, marker khong duoc cam nen lan chay sau
# chay lai duoc. RONG tu 21-08-2026: `n5_ocr_qwen` da bi BO HAN, tuc da xoa
# khoi `config/…_btc_images.yaml`. De ten no o day nua la sai cho — plan.stages
# khong con no, nen dong loc ben duoi thanh vo nghia va nguoi doc se tuong
# tang do van con trong hop dong cua bundle.
#              van thay no la viec con lai. Khac han voi viec xoa tang khoi yaml
#              — xem o mo ta ngay duoi.
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"

# ⚠️ MỌI NGƯỜI CHẠY BUNDLE NÀY PHẢI ĐỂ CÙNG MỘT NHÁNH.
#
# Notebook chỉ dùng nhánh này để clone `src/`. Hai người clone hai nhánh khác
# nhau là đổ HAI BẢN CODE khác nhau vào cùng một bundle — đúng thứ
# `merge.check_fingerprints` sinh ra để bắt, và §2.5 gọi là "năm người phải
# cùng phiên bản". Nó không ném lỗi lúc chạy; nó lộ ra ở bước gộp, sau nhiều giờ.
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

# Bản .lock thắng bản dải phiên bản — 5 người phải cùng phiên bản (§2.5).
req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-images.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

## Step 0b — Modal, phân công, việc đã xong

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, unload_models
import csv
from mfe.contracts.ids import frame_idx_from_pts
from mfe.offline.stages import Keyframe

# Vá đè hàm read_keyframes của hệ thống để bỏ qua lỗi làm tròn số phẩy động (float)
def read_keyframes_patched(map_csv, keyframes_dir):
    rows = list(csv.DictReader(map_csv.open(encoding="utf-8")))
    out = []
    for row in rows:
        n = int(row["n"])
        pts_time, fps = float(row["pts_time"]), float(row["fps"])
        frame_idx = int(row["frame_idx"])
        out.append(Keyframe(n, pts_time, fps, frame_idx, keyframes_dir / f"{n:04d}.jpg"))
    if [kf.n for kf in out] != list(range(1, len(out) + 1)):
        raise ValueError(f"{map_csv.name}: cot n phai chay 1..N lien tuc, khong ho")
    return out


from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")       # private Kaggle Datasets, read-only
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
# `plan.stages` la HOP DONG cua bundle; `SKIP_STAGES` chi la lua chon cua LUOT
# CHAY NAY. Giu hai thu tach nhau: xoa tang khoi yaml se doi ca thu `merge` doi.

# --- VÁ TQDM PROGRESS BAR CHO N4_SIGLIP ---
def stage_n4_siglip_patched(ctx):
    import torch
    from PIL import Image
    from transformers import AutoModel, AutoProcessor
    from tqdm.auto import tqdm
    import numpy as np
    from mfe.offline.stages import cached_model, _siglip_feature_tensor, Stopwatch
    
    checkpoint = "google/siglip-so400m-patch14-384"
    batch_size = ctx.params.get("batch_size", 16)
    
    with Stopwatch() as timer:
        processor, model = cached_model(
            checkpoint,
            lambda: (
                AutoProcessor.from_pretrained(checkpoint),
                AutoModel.from_pretrained(checkpoint).eval().cuda(),
            ),
        )
        vectors = []
        images = ctx.images()
        
        # Thêm tqdm progress bar ở đây
        for start in tqdm(range(0, len(images), batch_size), desc=f"  [SigLIP] {ctx.video_id}", leave=False, unit="batch"):
            batch = [Image.open(p).convert("RGB") for p in images[start : start + batch_size]]
            inputs = processor(images=batch, return_tensors="pt").to("cuda")
            with torch.no_grad():
                features = _siglip_feature_tensor(model.get_image_features(**inputs))
            vectors.append(features.float().cpu().numpy())
            
    return ctx.writer.write_vector_part(
        "siglip_vectors",
        ctx.video_id,
        ctx.ids(),
        np.concatenate(vectors),
        model=checkpoint,
        params={"batch_size": batch_size, "tower": "image"},
        elapsed_s=timer.seconds,
    )

STAGES["n4_siglip"] = stage_n4_siglip_patched
# ----------------------------------------

STAGE_LIST = ["n4_siglip"]  # Đã khoá cứng chỉ chạy SigLIP, tuyệt đối không đụng OCR
PARAMS     = RAW.get("params", {})
EXTRAS     = RAW["extras"]

MY_ZIPS   = plan.zips_for(MEMBER)
MY_VIDEOS = plan.videos_for(MEMBER)
status    = StageStatus.load(transport, layout, MEMBER)

# Điều kiện tiên quyết: notebook 02 phải xong TRƯỚC. Thiếu `n0_import_btc` thì
# `map-keyframes` của video đó chưa lên Volume, và bundle sẽ có vector cho một
# bộ keyframe mà chính nó không mô tả.
todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"zip     : {', '.join(n for n, _ in MY_ZIPS)}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")
if ONLY_VIDEOS:
    print(f"CHẾ ĐỘ THỬ : chỉ {len(ONLY_VIDEOS)} video -> {', '.join(ONLY_VIDEOS)}")
if REDO_STAGES:
    print(f"CHẠY LẠI   : {', '.join(REDO_STAGES)} (ghi đè part + marker cũ)")


In [ ]:
import shutil, zipfile
from concurrent.futures import ThreadPoolExecutor
import modal

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30

# Hàm kéo ảnh cực tốc độ từ Modal bằng Đa luồng
def download_modal_dir(vol, remote_dir, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)
    
    def download_file(entry):
        if entry.type == modal.volume.FileEntryType.FILE:
            dest = local_dir / entry.path.split('/')[-1]
            with dest.open("wb") as f:
                for chunk in vol.read_file(entry.path):
                    f.write(chunk)
                    
    try:
        entries = list(vol.listdir(remote_dir))
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy thư mục trên Modal: {remote_dir}") from e
    
    with ThreadPoolExecutor(max_workers=22) as pool:
        pool.map(download_file, entries)
        
    files = [e for e in entries if e.type == modal.volume.FileEntryType.FILE]
    return local_dir, len(files)

# Các hàm liên quan tới ZIP (Kaggle Dataset) của bản cũ đã bị loại bỏ vì dùng trực tiếp Modal
def download(*args, **kwargs): pass
MAPS_DIR = None
print(f"đĩa trống: {free_gb():.1f} GB")


## Step 1 — vòng lặp

Theo **video**, không theo tầng: model đã nằm trong cache nên số lần nạp giống hệt cách chia theo
tầng, nhưng mỗi video chỉ cần ~35 MB ảnh trên đĩa thay vì cả gói đã bung.

Marker cắm theo từng *(tầng, video)*. Phiên đứt giữa chừng thì lần chạy sau bỏ đúng những việc đã
xong, kể cả khi nó đứt giữa hai tầng của cùng một video.

`merge=True` khi ghi manifest: bundle này được đổ bởi **ba notebook** (02 nhập, 03 đây, 04 video)
và cả ba cùng ghi `part-manifest/{video}.json`. Không gộp thì mỗi lượt xoá sạch artifact của lượt
trước, còn marker vẫn báo "xong" nên không lần chạy nào phục hồi lại.

In [ ]:
import time, traceback
import modal

def stages_for(video_id):
    return [s for s in STAGE_LIST
            if s in REDO_STAGES or not status.is_done(s, video_id)]

failures = []
started = time.perf_counter()
done_count = 0

def run_one(video_id, kf_dir, map_csv, stages):
    keyframes = read_keyframes_patched(map_csv, kf_dir)
    ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, layout, WORK)
    markers = []
    for stage in stages:
        ctx.params = dict(PARAMS.get(stage, {}))
        marker = STAGES[stage](ctx)
        markers.append(marker)
        status.done.add((stage, video_id))
        print(f"      {stage:<16} {marker.get('rows', '-'):>6} dòng  {marker.get('elapsed_s')}s")
    return keyframes, markers

print("Kết nối Modal Volume để kéo ảnh trực tiếp...")
vol = modal.Volume.from_name(VOLUME)

mine = set(MY_VIDEOS)
if ONLY_VIDEOS:
    mine &= set(ONLY_VIDEOS)
    
pending = [v for v in MY_VIDEOS if v in mine and stages_for(v)]

if not pending:
    print(f"=== Không còn việc cần chạy cho {MEMBER} ===")
else:
    print(f"\n=== Còn {len(pending)}/{len(MY_VIDEOS)} video cần chạy SigLIP ===")

for video_id in pending:
    stages = stages_for(video_id)
    try:
        remote_kf_dir = f"bundle/{plan.bundle}/keyframes/{video_id}"
        kf_dir = WORK / "keyframes" / video_id
        
        # Tải thẳng ảnh giải nén sẵn từ Modal về thư mục tạm
        _, n_img = download_modal_dir(vol, remote_kf_dir, kf_dir)
        
        # Tải CSV mapping từ Modal
        remote_csv = f"bundle/{plan.bundle}/map-keyframes/{video_id}.csv"
        map_csv = WORK / "map-keyframes" / f"{video_id}.csv"
        map_csv.parent.mkdir(parents=True, exist_ok=True)
        with map_csv.open("wb") as f:
            for chunk in vol.read_file(remote_csv):
                f.write(chunk)
                                   
        print(f"  {video_id}  {n_img} ảnh  ({len(stages)} tầng)")
        keyframes, markers = run_one(video_id, kf_dir, map_csv, stages)
        writer.write_part_manifest(video_id, markers, keyframes=len(keyframes), merge=True)
        
        done_count += 1
        print(f"  [{done_count:>3}/{len(pending)}] {video_id} xong")
    except Exception as err:
        failures.append((video_id, repr(err)))
        print(f"  [LỖI] {video_id}: {err}")
        traceback.print_exc(limit=3)
    finally:
        shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
        (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)
        if not KEEP_MODELS:
            unload_models()

print("\n" + "=" * 72)
print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")


## Step 2 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print()
    print(f"còn {len(left)} việc — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print()
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")